# 11 — Baseline variable ablation

In [ ]:
from pathlib import Path
import sys
import warnings
warnings.filterwarnings("ignore")

PROJECT_ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                    if (p / "analysis" / "lib").exists())
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Patch, ConnectionPatch
from joblib import Parallel, delayed
from sklearn.metrics import average_precision_score, roc_auc_score

from analysis.lib.data import restricted_pool, build_demographics_matrix
from analysis.lib.pipelines import make_baseline_only
from analysis.lib.cv import pooled_cv
from analysis.lib.plotting import apply_manuscript_style, VARIABLE_LABELS_SHORT

DATA_DIR = PROJECT_ROOT / "data"
LACUNA   = DATA_DIR / "lacuna" / "METAVCI_PSCI_LACUNA" / "derivatives"
OUT_DIR  = PROJECT_ROOT / "outputs" / "predictive_modeling_core" / "pooled"
FIG_OUT  = PROJECT_ROOT / "outputs" / "manuscript_figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_OUT.mkdir(parents=True, exist_ok=True)

CLIN_PATH = DATA_DIR / "tabular" / "clinical_data.csv"
assert CLIN_PATH.exists(), f"missing {CLIN_PATH} — see data/README.md"

TARGETS = [
    "cognition-PSCI_present",
    "cognition-aef_domain_score",
    "cognition-ps_domain_score",
    "cognition-language_domain_score",
    "cognition-verbalmemory_domain_score",
    "cognition-vsfunctions_domain_score",
    "cognition-vsmemory_domain_score",
]

In [ ]:
# Compute (cached on output CSV)
def _one_fold(fold_idx, tr, te, X, y):
    try:
        pipe = make_baseline_only()
        pipe.fit(X[tr], y[tr])
        s_tr = pipe.decision_function(X[tr])
        s_te = pipe.decision_function(X[te])
        single_tr = len(np.unique(y[tr])) < 2
        single_te = len(np.unique(y[te])) < 2
        return {"Fold": fold_idx,
                "AUPRC": float("nan") if single_te else float(average_precision_score(y[te], s_te)),
                "AUROC": float("nan") if single_te else float(roc_auc_score(y[te], s_te)),
                "AUPRC_train": float("nan") if single_tr else float(average_precision_score(y[tr], s_tr)),
                "AUROC_train": float("nan") if single_tr else float(roc_auc_score(y[tr], s_tr))}
    except Exception as exc:
        warnings.warn(f"fold {fold_idx} failed: {exc}")
        return {"Fold": fold_idx, "AUPRC": float("nan"), "AUROC": float("nan"),
                "AUPRC_train": float("nan"), "AUROC_train": float("nan")}


SCORES_CSV = OUT_DIR / "baseline_ablation_scores.csv"
SUMMARY_CSV = OUT_DIR / "baseline_ablation_summary.csv"

if SCORES_CSV.exists():
    print(f"[skip] {SCORES_CSV} exists")
else:
    clin_all = pd.read_csv(CLIN_PATH, index_col=0)
    pool = restricted_pool(LACUNA, clin_all, ["acuteinfarct"])
    clinical = clin_all.loc[pool].copy()
    print(f"acute pool: N = {len(pool)}")

    baseline_cols = [c for c in clinical.columns
                     if c.startswith(("demographics-", "stroke-", "vascularrisk-"))]
    cv = pooled_cv(seed=42)
    rows = []
    for target in TARGETS:
        y_series = clinical[target]
        valid = y_series.notna()
        y = y_series[valid].values.astype(int)
        if y.sum() < 10 or len(y) - y.sum() < 10:
            print(f"[skip] {target}")
            continue
        sub = clinical[valid].copy()
        print(f"=== {target} (n={len(y)}) ===")
        for ablate in [None] + baseline_cols:
            df_a = sub.drop(columns=[ablate]) if ablate in sub.columns else sub
            X, names = build_demographics_matrix(df_a, exclude_cohort=False)[:2]
            label = "full" if ablate is None else ablate
            fold_results = Parallel(n_jobs=-1)(
                delayed(_one_fold)(i, tr, te, X, y)
                for i, (tr, te) in enumerate(cv.split(X, y)))
            for r in fold_results:
                rows.append({"Target": target, "AblatedVariable": label,
                             "Fold": r["Fold"], "AUPRC": r["AUPRC"],
                             "AUROC": r["AUROC"], "AUPRC_train": r["AUPRC_train"],
                             "AUROC_train": r["AUROC_train"],
                             "n_features": int(X.shape[1])})
            print(f"  {label:<40s} AUPRC={np.nanmean([r['AUPRC'] for r in fold_results]):.4f}")

    df = pd.DataFrame(rows)
    df.to_csv(SCORES_CSV, index=False)

    agg = (df.groupby(["Target", "AblatedVariable"])
           .agg(AUPRC_mean=("AUPRC", "mean"), AUPRC_sd=("AUPRC", "std"),
                AUROC_mean=("AUROC", "mean"), AUROC_sd=("AUROC", "std"),
                n_features=("n_features", "first"), n_folds=("AUPRC", "count"))
           .reset_index())
    full = (agg[agg["AblatedVariable"] == "full"].set_index("Target")
            [["AUPRC_mean", "AUROC_mean"]]
            .rename(columns={"AUPRC_mean": "AUPRC_full", "AUROC_mean": "AUROC_full"}))
    agg = agg.merge(full, left_on="Target", right_index=True, how="left")
    agg["dAUPRC"] = agg["AUPRC_mean"] - agg["AUPRC_full"]
    agg["dAUROC"] = agg["AUROC_mean"] - agg["AUROC_full"]
    agg.to_csv(SUMMARY_CSV, index=False)
    print(f"saved {SCORES_CSV} and {SUMMARY_CSV}")

In [ ]:
# Figure: ΔAUPRC heatmap + attached mean-Δ bars
apply_manuscript_style()

VAR_LABELS = {
    "demographics-age":               "Age",
    "demographics-sex":               "Sex",
    "demographics-ethnicity":         "Ethnicity",
    "demographics-education_STROKOG": "Education",
    "demographics-cohort":            "Cohort",
    "stroke-handedness":              "Handedness",
    "stroke-NIHSS":                   "NIHSS",
    "stroke-prior_stroke":            "Prior stroke",
    "stroke-prior_TIA":               "Prior TIA",
    "stroke-interval_onset_NPO_days": "Onset → NPA (days)",
    "stroke-interval_onset_MRI_days": "Onset → MRI (days)",
    "stroke-total_infarct_volume_ml": "Infarct volume",
    "vascularrisk-obesity":           "Obesity",
    "vascularrisk-smoking_present":   "Current smoker",
    "vascularrisk-smoking_past":      "Past smoker",
    "vascularrisk-hypertension":      "Hypertension",
    "vascularrisk-hyperlipidemia":    "Hyperlipidemia",
    "vascularrisk-diabetes_mellitus": "Diabetes",
}
GROUP_COLOR = {"demographics-": "#3D5A8C", "stroke-": "#C75A4A",
               "vascularrisk-": "#7FB069"}


def _group_color(var):
    for prefix, color in GROUP_COLOR.items():
        if var.startswith(prefix):
            return color
    return "#888"


summary = pd.read_csv(SUMMARY_CSV)
mat = (summary[summary["AblatedVariable"] != "full"]
       .pivot(index="AblatedVariable", columns="Target", values="dAUPRC"))
mat = mat.reindex(columns=[t for t in TARGETS if t in mat.columns])
mat = mat.loc[[v for v in mat.index if v in VAR_LABELS]]
rank = mat.mean(axis=1).sort_values()
mat = mat.loc[rank.index]
n_rows, n_cols = mat.shape

fig = plt.figure(figsize=(15.5, 10))
gs = GridSpec(1, 2, figure=fig, wspace=0.03, width_ratios=[3.0, 0.9],
              left=0.045, right=0.845, top=0.92, bottom=0.20)
ax = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])

vmax = float(np.nanmax(np.abs(mat.values)))
im = ax.imshow(mat.values, cmap="RdBu", vmin=-vmax, vmax=vmax, aspect="auto")
contrast = vmax * 0.55
for i in range(n_rows):
    for j in range(n_cols):
        v = mat.values[i, j]
        if not np.isfinite(v):
            continue
        tc = "white" if abs(v) > contrast else "#222"
        ax.text(j, i, f"{v:+.3f}", ha="center", va="center",
                fontsize=9, fontweight="bold", color=tc)
ax.set_xticks(range(n_cols))
ax.set_xticklabels([VARIABLE_LABELS_SHORT.get(c, c) for c in mat.columns],
                   rotation=25, ha="right", fontsize=11, fontweight="bold")
ax.set_yticks(range(n_rows))
ax.tick_params(axis="y", left=False, labelleft=False)

means = mat.mean(axis=1).values
sds = mat.std(axis=1).values
ys = np.arange(n_rows)
colors = [_group_color(v) for v in mat.index]
ax2.barh(ys, means, xerr=sds, height=0.72, color=colors,
         edgecolor="black", linewidth=0.5, alpha=0.92,
         error_kw={"linewidth": 0.7, "alpha": 0.6})
ax2.axvline(0, color="#222", lw=0.8)
for i, y in enumerate(ys):
    left_edge = min(0, means[i] - sds[i])
    con = ConnectionPatch(xyA=(n_cols - 0.5, y), xyB=(left_edge, y),
                          coordsA="data", coordsB="data", axesA=ax, axesB=ax2,
                          color="lightgray", linestyle="--", linewidth=0.8,
                          alpha=0.6, zorder=1.5, shrinkB=4)
    ax2.add_artist(con)

ax2.set_ylim(n_rows - 0.5, -0.5)
ax2.set_yticks(ys)
ax2.yaxis.tick_right()
ax2.set_yticklabels([VAR_LABELS[v] for v in mat.index], fontsize=10, fontweight="bold")
ax2.tick_params(axis="y", length=0, pad=6)
for lbl, v in zip(ax2.get_yticklabels(), mat.index):
    lbl.set_color(_group_color(v))
ax2.set_xlabel("Mean Δ AUPRC\nacross 7 targets", fontsize=11, fontweight="bold")
ax2.grid(True, axis="x", alpha=0.25, ls=":")
for sp in ("top", "left", "right"):
    ax2.spines[sp].set_visible(False)

cax = fig.add_axes([0.24, 0.055, 0.20, 0.018])
cb = fig.colorbar(im, cax=cax, orientation="horizontal")
cb.set_label("Δ AUPRC  (negative = drop hurts)", fontsize=10,
             fontweight="bold", labelpad=6)
cb.ax.tick_params(labelsize=9)

handles = [Patch(color=GROUP_COLOR["demographics-"], label="Demographics"),
           Patch(color=GROUP_COLOR["stroke-"], label="Stroke"),
           Patch(color=GROUP_COLOR["vascularrisk-"], label="Vascular risk")]
fig.legend(handles=handles, loc="center left", bbox_to_anchor=(0.53, 0.062),
           ncol=3, frameon=False, prop={"size": 10, "weight": "bold"})

out_png = FIG_OUT / "suppl_baseline_variable_ablation.png"
fig.savefig(out_png, dpi=220, bbox_inches="tight")
fig.savefig(out_png.with_suffix(".svg"), bbox_inches="tight")
fig.savefig(out_png.with_suffix(".pdf"), bbox_inches="tight")
print(f"saved {out_png}")